# 🏃 Multi-Agent Marathon Planner

## Workshop práctico: ADK, A2A y evaluación con Vertex AI

En este workshop vas a construir y ejecutar un sistema multiagente que transforma una intención humana —organizar una maratón urbana— en un plan operativo, evaluado y simulado.

No vas a limitarte a ejecutar celdas. En cada paso vas a identificar una decisión de arquitectura, observar su efecto y comprobarla con una salida verificable.

### Al finalizar vas a poder

- definir responsabilidades claras entre agentes especializados;
- combinar delegación local con comunicación remota mediante A2A;
- separar razonamiento probabilístico de cálculos determinísticos;
- evaluar respuestas con rúbricas explícitas y criterios ponderados;
- observar una ejecución multiagente a través de eventos, logs y una interfaz;
- diseñar rutas de degradación cuando un modelo o servicio remoto no está disponible.

---

## Antes de empezar

| # | Requisito | Cómo verificarlo |
|---|---|---|
| 1 | Una cuenta Google con acceso a un proyecto de Google Cloud | [Abrí la consola](https://console.cloud.google.com) y confirmá que el selector muestra un proyecto |
| 2 | Facturación habilitada | En *Facturación*, confirmá que el proyecto tiene una cuenta vinculada. Vertex AI requiere billing y genera consumo facturable |
| 3 | Permisos suficientes | Tu usuario debe ser Owner, Editor o tener el rol *Vertex AI User* |
| 4 | El **Project ID** | Copiá el identificador, no el nombre visible; por ejemplo `mi-proyecto-123456` |
| 5 | Una copia propia del notebook | En Colab: *Archivo → Guardar una copia en Drive* |

Todo se ejecuta en la VM temporal de Colab: dependencias, autenticación y servidores locales. No necesitás instalar herramientas en tu computadora.

> **Importante:** prepará el proyecto y la facturación antes de la sesión. Es la única parte que no conviene resolver durante el workshop.

---

## Arquitectura que vamos a construir

Cuatro agentes especializados, distribuidos en dos procesos, convierten el pedido en un plan evaluado y una experiencia de carrera simulada:

```text
                        ┌──────────────────────┐
   pedido humano ─────► │    planner_agent     │
                        │     orquestador      │
                        └───┬──────────────┬───┘
            AgentTool       │              │      A2A / JSON-RPC :8089
       mismo proceso        │              │      proceso remoto
                        ┌───▼──────┐   ┌───▼──────────────────┐
                        │evaluator │   │ simulator_agent      │
                        │  agent   │   │ gate de readiness    │
                        │7 métricas│   └──────────┬───────────┘
                        └──────────┘              │ AgentTool
                                           ┌─────▼────────────┐
                                           │ runner_agent     │
                                           │ cuatro cohortes  │
                                           └──────────────────┘
```

### Responsabilidades

1. **Planner:** transforma la intención en un plan operativo y coordina la delegación.
2. **Evaluator:** puntúa la calidad con siete criterios; no diseña ni simula.
3. **Simulator:** determina si el plan está listo para ejecutarse.
4. **Runner:** analiza la experiencia de cuatro cohortes: ritmo, congestión, hidratación, fatiga, abandono y llegada.

La arquitectura no busca “tener muchos chats”. Cada agente tiene una responsabilidad, herramientas concretas y un límite definido. El Runner representa cuatro cohortes; no ejecutamos 30.000 llamadas a un modelo. Esa decisión conserva el valor de la simulación con costo y latencia controlados.

### Cómo usar este notebook

Cada bloque sigue la misma secuencia:

1. **Comprender:** leé el objetivo y la decisión de diseño.
2. **Predecir:** respondé la pausa de diseño antes de ejecutar.
3. **Ejecutar:** corré la celda una sola vez y esperá la señal de éxito.
4. **Observar:** relacioná la salida con el concepto del bloque.
5. **Experimentar:** modificá únicamente los parámetros indicados.

### Recorrido sugerido

| Paso | Resultado | Tiempo estimado |
|---|---|---:|
| 0 | Entorno reproducible, autenticado y validado | 5 min |
| 1 | Evaluador con siete criterios y fallback heurístico | 10 min |
| 2 | Ruta calculada sobre una red vial y visualizada | 8 min |
| 3 | Cuatro agentes conectados en dos procesos | 12 min |
| 4 | Ejecución end-to-end, traza y Race View | 20 min |
| 5 | Laboratorio de resiliencia | 5 min |

> 📦 Código y material complementario: https://github.com/jorgeucano/Multi-Agent-Marathon-Planner-Workshop


---
## Paso 0 — Preparar un entorno reproducible

Antes de trabajar con agentes necesitamos eliminar variables accidentales: versiones incompatibles, credenciales incompletas, modelos no disponibles y código desactualizado.

Este paso establece una base común para todo el grupo. Al finalizar, el preflight debe confirmar dependencias, autenticación, disponibilidad del modelo, puertos y archivos de Skills.

> **Resultado del paso:** un entorno validado en el que los errores posteriores pertenecen al sistema que estamos estudiando, no a la instalación.


**Antes de ejecutar 0.1 — Fijar versiones también es diseñar el sistema**

**Objetivo.** Crear un entorno reproducible antes de hablar con ningún agente.

En Python, `>=` no significa “compatible para siempre”. ADK, A2A y Vertex AI evolucionan por separado, y un cambio de versión mayor puede eliminar imports que el proyecto necesita. Por eso esta celda instala un rango probado y después verifica un símbolo concreto de A2A.

**Mientras corre, buscá esto:** no alcanza con que `pip` termine sin error; las versiones deben pertenecer a la línea esperada y `a2a.server.apps` debe importar.

> 🧭 **Pausa de diseño:** ¿qué diferencia hay entre que una dependencia esté instalada y que sea compatible con nuestra arquitectura?

✅ **Señal de éxito:** cuatro checks verdes y el mensaje `a2a.server.apps importa`. Si Colab pide reiniciar el entorno, reinicialo y repetí solamente esta celda.

In [ ]:
# @title Paso 0.1 — Instalar dependencias (~2 min) { display-mode: "form" }
# Versiones alineadas con pyproject.toml y límites superiores explícitos.
#
# Usar solo pisos (>=) puede resolver a google-adk 2.x,
# a2a-sdk 1.x y aiplatform 2.x, donde `a2a.server.apps` ya no existe y los
# servidores A2A no se pueden ni construir. Los techos son lo que hace que
# este notebook funcione. Ver docs/GOTCHAS.md #10 en el repo.

%pip install -q \
    "google-cloud-aiplatform[agent_engines,adk,evaluation]>=1.121.0,<2" \
    "google-adk>=1.25.0,<2" \
    "a2a-sdk>=0.3.9,<1" \
    "pydantic>=2.12.0" \
    "python-dotenv>=1.0.0" \
    "httpx>=0.27.0" \
    "uvicorn>=0.30.0" \
    "google-auth>=2.0.0" \
    "pandas>=2.0.0" \
    folium

import importlib.metadata as md

print("Instalado:")
for pkg, esperado in (("google-adk", "1."), ("a2a-sdk", "0."), ("google-cloud-aiplatform", "1."), ("pydantic", "2.")):
    try:
        v = md.version(pkg)
        ok = "✓" if v.startswith(esperado) else "✗ MAJOR INCORRECTO"
        print(f"  {ok} {pkg:28s} {v}")
    except md.PackageNotFoundError:
        print(f"  ✗ {pkg:28s} NO INSTALADO")

# El simbolo que desaparece en a2a-sdk 1.x: si esto importa, estamos en la linea correcta.
try:
    from a2a.server.apps import A2AStarletteApplication  # noqa: F401
    print("\n✓ a2a.server.apps importa — versiones compatibles con este workshop")
except ModuleNotFoundError:
    print("\n✗ a2a.server.apps NO existe — quedó instalado a2a-sdk 1.x. Reiniciá el runtime y volvé a correr esta celda.")

print("\n⚠️  Si Colab pide reiniciar el entorno, hacelo y volvé a ejecutar únicamente esta celda.")

**Antes de ejecutar 0.2 — Identidad, región y modelo: tres decisiones distintas**

**Objetivo.** Conectar el notebook con tu proyecto de Google Cloud y comprobar la conexión con una llamada mínima.

Esta celda hace tres cosas que suelen confundirse: autentica a la persona, selecciona el proyecto que paga y define dónde vive el modelo. `PROJECT_ID`, `LOCATION` y `MODEL` no son datos decorativos: juntos determinan si Vertex AI puede responder.

**Antes de ejecutar:** completá el ID real del proyecto. Para Gemini 3.x dejá `LOCATION=global`. Aceptá el permiso de Google cuando aparezca.

> 🧭 **Pausa de diseño:** si tenemos credenciales válidas pero elegimos una región donde el modelo no existe, ¿es un problema de autenticación o de despliegue?

✅ **Señal de éxito:** la última línea muestra que el modelo respondió `ok`. Un `403` suele indicar billing o permisos; un `404`, combinación incorrecta de modelo y región.

In [ ]:
# @title Paso 0.2 — Autenticación, proyecto y modelos { display-mode: "form" }
# Completá tu project id. El popup de Google te va a pedir permiso: aceptá.

PROJECT_ID = ""  # @param {type:"string"}
LOCATION = "global"  # @param ["global", "us-central1"]
MODEL = "gemini-3.1-flash-lite"  # @param ["gemini-3.1-flash-lite", "gemini-2.5-flash", "gemini-3-flash-preview"]

import os

assert PROJECT_ID, "Poné tu GOOGLE_CLOUD_PROJECT arriba y volvé a correr la celda."

from google.colab import auth
auth.authenticate_user(project_id=PROJECT_ID)
print("✓ Credenciales listas (esto reemplaza al `gcloud auth application-default login`)")

# La única API que hace falta.
!gcloud config set project {PROJECT_ID} 2>/dev/null
!gcloud services enable aiplatform.googleapis.com 2>&1 | tail -1

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "true"
# Un solo modelo flash para los cuatro agentes: baja latencia y fácil de
# comparar. También se pueden usar gemini-3-flash-preview o
# gemini-3.1-pro-preview; para esos modelos usá location=global.
for var in ("PLANNER_MODEL", "EVALUATOR_MODEL", "SIMULATOR_MODEL", "RUNNER_MODEL"):
    os.environ[var] = MODEL

print(f"✓ Proyecto: {PROJECT_ID}  ·  región: {LOCATION}  ·  modelo: {MODEL}")

# Chequeo real: ¿responde el modelo en esta región?
# Compatibilidad regional: los modelos Gemini 3.x configurados acá
# requieren `global`; una región no compatible devuelve 404.
from google import genai

client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)
try:
    r = client.models.generate_content(model=MODEL, contents="Respondé solo: ok")
    print(f"  ✓ {MODEL} responde en {LOCATION}: {r.text.strip()[:30]!r}")
except Exception as e:
    msg = str(e)
    print(f"  ✗ {MODEL} en {LOCATION}\n      {msg[:200]}")
    if "billing" in msg.lower() or msg.startswith("403"):
        print("      → Casi seguro el proyecto no tiene facturación habilitada. Ver la tabla 'Antes de empezar'.")
    else:
        print("      → probá LOCATION='global' o MODEL='gemini-2.5-flash' (ese existe en us-central1).")

**Antes de ejecutar 0.3 — Código correcto, configuración correcta y una prueba antes de empezar**

**Objetivo.** Traer la versión actual del workshop, escribir la configuración compartida y detectar problemas temprano.

El notebook es la guía; los agentes viven en el repositorio. Si la carpeta ya existe, la actualizamos a `origin/main` para evitar ejecutar código viejo sin darnos cuenta. Después, el preflight prueba imports, credenciales, modelos, puertos y archivos de Skills.

**Qué observar:** anotá el hash del commit. Es la evidencia de que todas las personas están usando la misma versión.

> 🧭 **Pausa de diseño:** ¿por qué conviene que un sistema falle en el preflight y no durante la delegación número tres?

✅ **Señal de éxito:** `0 failure(s)`. Los warnings de puertos indican procesos de una corrida anterior; los failures deben resolverse antes del Paso 1.

In [ ]:
# @title Paso 0.3 — Traer el código y correr el preflight { display-mode: "form" }

REPO_URL = "https://github.com/jorgeucano/Multi-Agent-Marathon-Planner-Workshop.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}

import os, subprocess, sys

REPO_DIR = "/content/marathon-agents"

if not REPO_URL:
    raise SystemExit("Falta REPO_URL: pegá arriba la URL del repo del workshop.")

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}
else:
    # Si ya existe, ACTUALIZARLO. Saltear esto es una trampa silenciosa: seguís
    # corriendo el código de la primera vez que clonaste y no hay ninguna señal.
    # fetch + reset (no pull): la historia del repo puede haber sido reescrita.
    print(f"{REPO_DIR} ya existe → actualizando a origin/{BRANCH}")
    !cd {REPO_DIR} && git fetch --tags --force origin && git reset --hard origin/{BRANCH}

assert os.path.isdir(REPO_DIR), "El clone falló. Revisá la URL y que el repo sea público."

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

# El .env que leen todos los agentes (misma config que os.environ de la celda 0.2).
with open(".env", "w") as f:
    for var in ("GOOGLE_CLOUD_PROJECT", "GOOGLE_CLOUD_LOCATION", "GOOGLE_GENAI_USE_VERTEXAI",
                "PLANNER_MODEL", "EVALUATOR_MODEL", "SIMULATOR_MODEL", "RUNNER_MODEL"):
        f.write(f"{var}={os.environ[var]}\n")
print("✓ .env escrito:\n")
!cat .env
print()

# Qué versión del código estás corriendo, para que nunca sea una sorpresa.
print("Commit:")
!git log --oneline -1
print()

# Si ya importaste los módulos del repo (celdas 1.x) ANTES de actualizar, Python
# tiene la versión vieja en memoria: reiniciá la sesión y volvé a correr.
import sys
if any(m.startswith("src.") for m in sys.modules):
    print("⚠️  Ya habías importado módulos del repo en esta sesión.")
    print("   Si el commit de arriba cambió: Entorno de ejecución → Reiniciar sesión,")
    print("   y volvé a correr desde la celda 0.1.\n")

# Los tags permiten recuperar el estado de cada etapa del workshop.
!git tag -l

# Preflight: detectar problemas antes de iniciar la arquitectura.
!python scripts/preflight.py

**Antes de ejecutar 0.4 — Recuperación opcional: actualizar una sesión existente**

Si ya habías corrido este notebook antes de una actualización del repo, en disco
tenés el código de la primera vez **y** en memoria los módulos ya importados.
Esta celda arregla las dos cosas de una, sin reiniciar la sesión. Es idempotente:
corrila las veces que quieras.

In [ ]:
# @title 0.4 — Actualizar el repo y purgar los módulos cacheados { display-mode: "form" }

import os, sys, subprocess

REPO_DIR = "/content/marathon-agents"

antes = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                       capture_output=True, text=True).stdout.strip()

# fetch + reset, NO pull: la historia del repo puede haber sido reescrita.
!cd {REPO_DIR} && git fetch --tags --force origin && git reset --hard origin/main

despues = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                         capture_output=True, text=True).stdout.strip()

# Actualizar archivos no sirve de nada si Python ya tiene el módulo cacheado:
# sacarlos de sys.modules hace que el próximo import lea el código nuevo.
purgados = [m for m in list(sys.modules) if m == "src" or m.startswith("src.")]
for m in purgados:
    del sys.modules[m]

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"\ncommit: {antes} → {despues}" + ("  (sin cambios)" if antes == despues else "  ✓ ACTUALIZADO"))
print(f"módulos purgados de la memoria: {len(purgados)}" + (f" → {', '.join(sorted(purgados)[:4])}..." if purgados else ""))
if antes != despues:
    print("\nVolvé a correr las celdas 1.x en adelante: ahora usan el código nuevo.")
    print("Si alguna sigue rara, Entorno de ejecución → Reiniciar sesión y empezá desde 0.1.")

---
## Paso 1 — Separar creación y evaluación

El Planner propone un plan. El Evaluator lo compara contra una rúbrica estable. Esta separación evita que el mismo agente sea autor y juez de su propia respuesta.

El Evaluator puntúa el plan en siete criterios. Seis utilizan métricas personalizadas de Vertex AI Evaluation; uno, `distance_compliance`, se calcula de forma determinística.

```python
builder = types.MetricPromptBuilder(
    metric_definition="Evaluar si la ruta mantiene acceso de emergencia...",
    criteria={
        "Emergency corridor access": "La ruta no bloquea hospitales sin un desvío documentado.",
        "Evacuation routes": "Las rutas de evacuación permanecen accesibles.",
    },
    rating_scores={
        "1": "Peligroso: corredores bloqueados sin desvíos",
        "50": "Preocupante: algunos accesos comprometidos",
        "100": "Seguro: acceso de emergencia mantenido",
    },
)
```

**Antes de ejecutar 1.1 — Rúbricas explícitas y pesos**

1. **Consistencia:** la escala define qué significa cada nivel.
2. **Trazabilidad:** cada puntuación incluye una explicación.
3. **Comparabilidad:** distintas versiones del plan se evalúan con el mismo criterio.
4. **Gobernanza:** un hallazgo crítico puede bloquear la aprobación aunque el promedio sea alto.

En las próximas celdas vas a inspeccionar los pesos, comprobar la regla de aprobación y comparar dos motores de evaluación que conservan el mismo contrato de salida.

> **Resultado del paso:** entender por qué evaluar no es volver a preguntar al mismo modelo si su respuesta “está bien”.


In [ ]:
# @title 1.1 — Los 7 criterios y sus pesos { display-mode: "form" }
# Nada de red todavía: esto es leer la config del Evaluator.

from src.planner_agent.evaluator.config import (
    CRITERION_WEIGHTS, SEVERITY_THRESHOLDS, MODEL as EVALUATOR_MODEL,
)

print(f"Modelo juez: {EVALUATOR_MODEL}\n")
print(f"{'criterio':28s} {'peso':>6s}")
print("-" * 55)
for criterio, peso in sorted(CRITERION_WEIGHTS.items(), key=lambda kv: -kv[1]):
    barra = "█" * int(peso * 100 / 2.5)
    print(f"{criterio:28s} {peso:>5.0%}   {barra}")
print("-" * 55)
print(f"{'TOTAL':28s} {sum(CRITERION_WEIGHTS.values()):>5.0%}\n")

print("Severidad por score:", SEVERITY_THRESHOLDS)
print("\nAprobación: overall_score >= 85  Y  cero hallazgos de severidad 'high'.")
print("Safety y logistics pesan 20% cada uno: 40% de la puntuación evalúa")
print("si el evento puede ejecutarse con seguridad y capacidad operativa.")

**Antes de ejecutar 1.2 — La aritmética del veredicto**

La puntuación final es un promedio ponderado, pero el promedio no alcanza para aprobar:

```python
overall_score = sum(scores[c] * w for c, w in CRITERION_WEIGHTS.items())
passed = overall_score >= 85.0 and not any(
    finding["severity"] == "high" for finding in findings
)
```

Un plan puede obtener un promedio superior a 85 y aun así ser rechazado si contiene un hallazgo de severidad alta. En un evento masivo, un resultado excelente en seis dimensiones no compensa un riesgo crítico de seguridad.

La próxima celda construye dos escenarios sin consumir tokens y permite observar la diferencia entre **promedio** y **regla de aprobación**.

> 🧭 **Pausa de diseño:** ¿qué otras condiciones deberían funcionar como veto, independientemente del promedio?


In [ ]:
# @title 1.2 — Un promedio alto NO alcanza (sin llamar a ningún modelo) { display-mode: "form" }

from src.planner_agent.evaluator.tools import _build_result

def veredicto(titulo, scores):
    r = _build_result(scores, {}, eval_method="demo")
    print(f"\n{titulo}")
    print(f"  overall_score : {r['overall_score']}")
    print(f"  passed        : {r['passed']}")
    for f in r["findings"]:
        print(f"    [{f['severity']:6s}] {f['criterion']}")
    return r

# Escenario A: todo bien.
veredicto("A) Plan sólido en todo", {c: 90.0 for c in CRITERION_WEIGHTS})

# Escenario B: casi todo excelente, pero seguridad colapsa.
scores_b = {c: 95.0 for c in CRITERION_WEIGHTS}
scores_b["safety_compliance"] = 30.0
veredicto("B) 95 en todo, 30 en seguridad", scores_b)

print("\n" + "=" * 60)
print("El escenario B saca un promedio altísimo y IGUAL no pasa.")
print("Un hallazgo 'high' es un veto, no un descuento.")
print("=" * 60)

**Antes de ejecutar 1.3 — Evaluación híbrida: calidad y continuidad**

`evaluate_plan` expone un solo contrato y admite dos estrategias:

```python
if project_id and eval_mode != "heuristic":
    try:
        scores, details = await _run_custom_eval(...)
        return _build_result(scores, details, "vertex_ai_eval")
    except Exception as error:
        logger.warning("Vertex AI Evaluation no está disponible: %s", error)

scores, details = _heuristic_eval(...)
return _build_result(scores, details, "heuristic")
```

La estrategia principal utiliza seis jueces LLM y un control determinístico. La alternativa heurística es más limitada, pero mantiene disponible el flujo y conserva el esquema de respuesta.

Una degradación responsable debe ser visible: el campo `eval_method` permite saber qué estrategia produjo el resultado. Primero vamos a ejecutar la alternativa heurística para conocer el contrato; después activaremos el juez real.

> 🧭 **Pausa de diseño:** ¿qué decisiones permitirías tomar con el fallback y cuáles exigirían obligatoriamente la evaluación principal?


In [ ]:
# @title 1.3 — Camino heurístico: instantáneo, cero tokens { display-mode: "form" }

import json, os

PLAN_FLOJO = "Vamos a hacer una maratón en Buenos Aires. Va a estar buena."

PLAN_COMPLETO = """
Maratón de Buenos Aires. Recorrido de 26.2 miles (42.195 km) con largada y
llegada en el Obelisco: Plaza de Mayo, Puerto Madero, Reserva Ecológica,
La Boca, San Telmo, Recoleta, Bosques de Palermo y Barrancas de Belgrano.
Water station cada 2.5 km, medical tent cada 5 km con ambulancia después del
km 30, chip timing en start line y finish line, emergency vehicle crossings
cada 2 miles con desvío señalizado alrededor del Hospital Argerich y el
Hospital Fernández. Cheer zones con community engagement en 4 barrios.
Budget de USD 2.74M con revenue de inscripciones, sponsors y expo.
Landmarks escénicos: Obelisco, Puente de la Mujer, Caminito, Floralis Genérica.
Post-race: medals, comida y recovery area en los Bosques de Palermo.
"""
os.environ["EVAL_MODE"] = "heuristic"   # fuerza el fallback
from src.planner_agent.evaluator.tools import evaluate_plan

for nombre, plan in [("PLAN FLOJO", PLAN_FLOJO), ("PLAN COMPLETO", PLAN_COMPLETO)]:
    r = await evaluate_plan(json.dumps({
        "user_intent": "Maratón escénica en Buenos Aires para 30.000 personas",
        "proposed_plan": plan,
    }))
    print(f"\n{'=' * 60}\n{nombre}  →  método: {r['eval_method']}")
    print(f"  overall_score: {r['overall_score']}   passed: {r['passed']}")
    for c, s in sorted(r["scores"].items(), key=lambda kv: kv[1]):
        print(f"    {c:28s} {s:6.1f}")
    if r["improvement_suggestions"]:
        print("  sugerencias:")
        for s in r["improvement_suggestions"][:3]:
            print(f"    · {s}")

print("\n→ Mismo código, mismo schema de salida. Solo cambia de dónde salen los números.")

**Antes de ejecutar 1.4 — Pasar del fallback al juez real**

**Objetivo.** Ejecutar exactamente el mismo contrato de evaluación, pero reemplazando las heurísticas por seis jueces LLM de Vertex AI Evaluation.

La comparación es importante: la interfaz de `evaluate_plan` no cambia. Lo que cambia es el mecanismo que produce seis de los siete scores. La distancia continúa siendo determinística. Este diseño permite degradar el servicio sin obligar al Planner a entender dos formatos de respuesta.

**Qué observar:** los criterios deberían obtener valores diferentes y cada finding debería traer una explicación. Seis `50.0` repetidos son una señal de evaluación fallida, no de consenso.

> 🧭 **Pausa de diseño:** ¿qué ganamos manteniendo el mismo esquema cuando cambiamos el motor de evaluación?

✅ **Señal de éxito:** `eval_method=vertex_ai_eval` y diversidad entre scores. Si aparece `heuristic`, leé el warning: el fallback funcionó, pero el juez real no.

In [ ]:
# @title 1.4 — Ahora el juez de verdad: Vertex AI Eval { display-mode: "form" }
# El mismo plan, puntuado por las 7 métricas reales: 6 con juez LLM y una
# determinística (la distancia, que es una regex). Con flash-lite tarda
# unos segundos; con un modelo de mayor capacidad puede tardar más.
#
# Validación: los criterios deberían producir puntuaciones diferentes. Seis 50.0
# y un overall de 52.5 indican que los jueces fallaron y el sistema los
# reemplazó por 50. Consultá docs/GOTCHAS.md, secciones 17 y 18.

import os, time, json

os.environ["EVAL_MODE"] = "auto"        # volvemos al camino Vertex AI Eval

t0 = time.time()
r = await evaluate_plan(json.dumps({
    "user_intent": "Maratón escénica en Buenos Aires para 30.000 personas",
    "proposed_plan": PLAN_COMPLETO,
}))
print(f"⏱  {time.time() - t0:.0f}s   método: {r['eval_method']}\n")

print(f"overall_score: {r['overall_score']}   passed: {r['passed']}")
print("-" * 70)
for c, s in sorted(r["scores"].items(), key=lambda kv: kv[1]):
    peso = CRITERION_WEIGHTS.get(c, 0)
    print(f"{c:28s} {s:6.1f}  × {peso:.0%}  = {s * peso:5.2f}")
print("-" * 70)

for f in r["findings"]:
    print(f"\n[{f['severity'].upper()}] {f['criterion']}")
    print(f"  {f['description'][:400]}")

if r["eval_method"] == "heuristic":
    print("\n⚠️  Se activó el fallback heurístico. Revisá el WARNING: puede ser")
    print("   cuota, región, o un juez que no devolvió JSON parseable.")
    print("   El sistema conservó disponibilidad mediante la estrategia alternativa.")
    print("   La respuesta debe identificarse como heurística y revisarse en consecuencia.")
else:
    distintos = len(set(r["scores"].values()))
    print(f"\n✓ {distintos} valores distintos entre 7 criterios.")
    if distintos <= 2:
        print("  ⚠️  Resultado atípico: seis valores 50.0 pueden indicar una falla no propagada.")

---
## Paso 2 — Separar razonamiento y cálculo

Un modelo puede decidir **cuándo** necesita una ruta, pero no debería inventar coordenadas, distancias ni cierres de calles. Esos datos deben provenir de una herramienta determinística.

El Planner descubre `plan_marathon_route` mediante el Skill `route-planning`:

```python
plan_marathon_route_func = load_tool_from_skill(
    "route-planning", "plan_marathon_route"
)
if plan_marathon_route_func:
    tools.append(FunctionTool(func=plan_marathon_route_func))
```

Esta frontera evita un modo de falla común: si una herramienta no se carga y el sistema no lo detecta, el modelo puede producir una respuesta plausible pero no verificable. Por eso el preflight confirma que el archivo y la función existen antes de iniciar los agentes.

La implementación del workshop utiliza una red vial por ciudad, Dijkstra para conectar landmarks, cierre de circuito y GeoJSON como formato de salida.

En 2.1 vas a inspeccionar tres evidencias:

- `network_source`: origen de la red utilizada;
- `raw_network_distance_km`: distancia recorrida sobre el grafo;
- `course_adjustment_km`: ajuste explícito para alcanzar 42.195 km.

En 2.2 vas a visualizar exactamente el mismo GeoJSON. El mapa es otra vista del resultado, no una segunda ruta.

> **Resultado del paso:** una ruta auditable cuya geometría no depende de la capacidad del LLM para generar texto convincente.


**Antes de ejecutar 2.1 — De una intención a geometría verificable**

**Objetivo.** Calcular una ruta de maratón sin pedirle coordenadas a un modelo de lenguaje.

La función carga una red vial conocida, ejecuta Dijkstra entre landmarks y cierra un circuito. El LLM puede decidir que necesita esta herramienta; no puede modificar la distancia que devuelve. Ésa es la frontera entre razonamiento probabilístico y cálculo determinístico.

**Qué observar:** `network_source`, distancia real sobre la red, ajuste de medición y cantidad de tramos que requieren desvío. Elegí otra ciudad para comprobar que la tool no está hardcodeada solamente para Buenos Aires.

> 🧭 **Pausa de diseño:** ¿qué parte debería decidir el agente y qué parte nunca debería inventar?

✅ **Señal de éxito:** 42.195 km, waypoints concretos y una lista de segmentos con cierres calculados.

In [ ]:
# @title 2.1 — Calcular la ruta (Dijkstra, sin LLM) { display-mode: "form" }

CIUDAD = "Buenos Aires"  # @param ["Buenos Aires", "Las Vegas", "Austin", "Tokyo"]

import importlib.util

spec = importlib.util.spec_from_file_location(
    "route_tools", "/content/marathon-agents/src/planner_agent/skills/route-planning/tools.py"
)
route_tools = importlib.util.module_from_spec(spec)
spec.loader.exec_module(route_tools)

ruta = route_tools.plan_marathon_route(CIUDAD)

print(f"Ciudad            : {ruta['city']}   (red: {ruta['network_source']})")
print(f"Largada/llegada   : {ruta['start_finish']}")
print(f"Distancia red     : {ruta['raw_network_distance_km']} km")
print(f"Ajuste de medición: {ruta['course_adjustment_km']:+} km")
print(f"Distancia oficial : {ruta['total_distance_km']} km / {ruta['total_distance_miles']} mi")
print(f"Waypoints         : {len(ruta['waypoints'])} ({ruta['unique_landmarks']} únicos)\n")

print("Primeros tramos:")
print(f"{'desde':32s} {'hasta':32s} {'km':>6s}  {'tipo':10s} {'corte':>5s}")
print("-" * 92)
for seg, cierre in list(zip(ruta["segments"], ruta["road_closures"]))[:8]:
    print(f"{seg['from'][:30]:32s} {seg['to'][:30]:32s} {seg['distance_km']:6.2f}  "
          f"{seg['road_type']:10s} {cierre['closure_severity']:>5d}")

print(f"\nTramos que necesitan desvío: "
      f"{sum(1 for c in ruta['road_closures'] if c['detour_required'])} de {len(ruta['road_closures'])}")
print("\n→ Esto es un número calculado, no una estimación de un modelo.")

**Antes de ejecutar 2.2 — Visualizar no significa recalcular**

**Objetivo.** Convertir el GeoJSON anterior en una evidencia visual que podamos inspeccionar.

Esta celda no pide otra ruta ni llama a Gemini. Consume el resultado de 2.1, invierte `[longitud, latitud]` a `[latitud, longitud]` para Folium y dibuja el circuito y sus landmarks. La visualización es una vista del mismo dato, no una segunda fuente de verdad.

**Qué observar:** largada y llegada coinciden, el loop sigue calles plausibles y las etiquetas corresponden a los waypoints impresos arriba.

> 🧭 **Pausa de diseño:** si el mapa se ve bonito pero la distancia calculada es incorrecta, ¿cuál de las dos evidencias manda?

✅ **Señal de éxito:** aparece el mapa interactivo con el circuito naranja y el marcador de largada/llegada.

In [ ]:
# @title 2.2 — Visualizar el GeoJSON en un mapa { display-mode: "form" }

import folium

coords = ruta["route_geojson"]["features"][0]["geometry"]["coordinates"]  # [lon, lat]
latlon = [[c[1], c[0]] for c in coords]

# Esri World Street Map: sin API key y sin bloqueo desde Colab.
# (OpenStreetMap devuelve 403 desde Colab por politica de uso; CartoDB pide key.)
m = folium.Map(
    location=latlon[0], zoom_start=13,
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles &copy; Esri",
)

folium.PolyLine(latlon, weight=6, opacity=0.85, color="#FF6B35",
                tooltip="26.2 mi / 42.195 km").add_to(m)

folium.Marker(latlon[0], tooltip=f"Largada / Llegada: {ruta['start_finish']}",
              icon=folium.Icon(color="green", icon="flag")).add_to(m)

# Un cartel por landmark único.
vistos = set()
for nombre in ruta["waypoints"]:
    if nombre in vistos:
        continue
    vistos.add(nombre)
    idx = ruta["waypoints"].index(nombre)
    folium.Marker(latlon[idx], tooltip=nombre, icon=folium.DivIcon(
        html=f'<div style="font:12px Arial;background:#fff;padding:2px 5px;'
             f'border-radius:3px;border:1px solid #999;white-space:nowrap">{nombre}</div>'
    )).add_to(m)

# Puestos de hidratación, ubicados proporcionalmente sobre el trazado.
agua = route_tools.add_water_stations(participants=30000)
for est in agua["stations"]:
    frac = est["km_mark"] / ruta["total_distance_km"]
    idx = min(int(frac * (len(latlon) - 1)), len(latlon) - 1)
    folium.CircleMarker(
        latlon[idx], radius=5, color="#0077B6", fill=True, fill_opacity=0.9,
        tooltip=f"{est['station_id']} · km {est['km_mark']} · {est['cups_required']:,} vasos",
    ).add_to(m)

m.fit_bounds(latlon)

print(f"{agua['water_station_count']} puestos de hidratación · "
      f"{agua['total_cups']:,} vasos · {agua['total_volunteers']} voluntarios")
med = route_tools.add_medical_tents(participants=30000)
print(f"{med['medical_tent_count']} puestos médicos · {med['total_medical_staff']} personas · "
      f"{med['ambulances']} ambulancias")
m

---
## Paso 3 — Conectar cuatro agentes en dos procesos

Hasta ahora ejecutamos componentes aislados. En este paso construimos la topología completa y observamos cómo cambia el sistema al cruzar una frontera de red.

| Mecanismo | Responsabilidad | Ubicación |
|---|---|---|
| `SkillToolset` | Cargar conocimiento procedural y herramientas | Proceso del Planner |
| `AgentTool(evaluator_agent)` | Delegar la evaluación | Proceso del Planner |
| `RemoteA2aAgent(...)` | Solicitar el veredicto de readiness | Proceso remoto del Simulator |
| `AgentTool(runner_agent)` | Analizar las cohortes | Proceso del Simulator |

La secuencia de delegación es:

```text
Planner ──AgentTool──► Evaluator
   │
   └────── A2A ─────► Simulator ──AgentTool──► Runner
```

A2A se utiliza donde necesitamos independencia de despliegue. `AgentTool` se utiliza donde queremos delegación semántica dentro del mismo proceso, sin introducir otra dependencia de red.

### Modelo de corredores

| Cohorte | Proporción | Aspecto observado |
|---|---:|---|
| Elite | 1% | Velocidad y disponibilidad temprana de servicios |
| Competitive | 14% | Ritmo sostenido y demanda de hidratación |
| Main pack | 60% | Congestión y capacidad operativa |
| Back of pack | 25% | Fatiga y cobertura médica prolongada |

Una función determinística calcula tamaños, ritmos, llegadas, abandono y riesgo. El Runner interpreta esas cifras y produce hallazgos operativos; no las modifica.

La topología se activa mediante configuración:

```python
if os.environ.get("SIMULATOR_AGENT_RESOURCE_NAME"):
    tools.append(create_simulation_controller_tool())
```

- `local:8089`: servidor del workshop;
- `projects/.../reasoningEngines/123`: despliegue en Agent Engine.

> **Resultado del paso:** el mismo Planner puede ejecutarse en modo local o conectarse con un agente remoto sin cambiar su lógica de negocio.


**Antes de ejecutar 3.1 — Levantar una frontera de red real**

**Objetivo.** Iniciar el proceso remoto que contiene al Simulator y al Runner.

Hasta ahora llamamos funciones dentro del notebook. Acá nace otro proceso, con su propio log y su propio puerto. El Simulator será accesible por A2A; el Runner seguirá siendo un sub-agente local dentro de ese proceso.

**Qué observar:** `Popen` devuelve enseguida, pero eso no significa que el servidor esté listo. `esperar_puerto` convierte readiness en una condición verificable.

> 🧭 **Pausa de diseño:** ¿por qué “el proceso existe” y “el servicio está listo” no son la misma cosa?

✅ **Señal de éxito:** `Simulator escuchando en :8089`. Si falla, la propia celda muestra el final del log.

In [ ]:
# @title 3.1 — Levantar el Simulation Controller (:8089) { display-mode: "form" }
# En Colab lo ejecutamos como un segundo proceso dentro de la misma VM.
# El proceso contiene dos agentes: Simulator + Runner.

import os, socket, subprocess, time

REPO_DIR = "/content/marathon-agents"
LOGS = "/content/logs"
os.makedirs(LOGS, exist_ok=True)

def puerto_vivo(port, timeout=0.4):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(timeout)
        return s.connect_ex(("127.0.0.1", port)) == 0

def esperar_puerto(port, nombre, log_path, segundos=90):
    for i in range(segundos):
        if puerto_vivo(port):
            print(f"✓ {nombre} escuchando en :{port}  ({i}s)")
            return True
        time.sleep(1)
    print(f"✗ {nombre} no levantó en {segundos}s. Últimas líneas del log:\n")
    print(open(log_path).read()[-2500:])
    return False

env = {**os.environ, "PYTHONUNBUFFERED": "1"}
sim_log = f"{LOGS}/simulator.log"

simulador = subprocess.Popen(
    ["python", "-m", "src.simulator_agent.runtime.local_server"],
    cwd=REPO_DIR, env=env,
    stdout=open(sim_log, "w"), stderr=subprocess.STDOUT,
)
print(f"Simulator arrancando (pid {simulador.pid})...")
esperar_puerto(8089, "Simulator", sim_log)

**Antes de ejecutar 3.2 — Descubrimiento antes que ejecución**

**Objetivo.** Inspeccionar el contrato público con el que otro agente descubre al Simulator.

La agent card describe identidad, capacidades, skills y endpoint. El Planner no necesita importar el código del Simulator: necesita conocer este contrato. Eso permite mover el servicio a otra máquina sin reescribir la lógica del orquestador.

**Qué observar:** nombre del agente, URL y skills anunciadas. Prestá atención a la ruta estándar `/.well-known/agent-card.json`.

> 🧭 **Pausa de diseño:** ¿qué información mínima necesita un agente para colaborar con otro que nunca importó?

✅ **Señal de éxito:** obtenemos JSON legible desde el puerto 8089; un `404` en otra ruta no significa que A2A esté roto.

In [ ]:
# @title 3.2 — La agent card del Simulator: lo que el Planner va a leer { display-mode: "form" }
# Antes de conectar los agentes, inspeccionamos el contrato publicado.

import json, httpx

card = httpx.get("http://127.0.0.1:8089/.well-known/agent-card.json", timeout=10).json()

print(json.dumps(card, indent=2)[:1200])
print("\n" + "=" * 60)
print("Ruta estándar: /.well-known/agent-card.json")
print("La ruta /.well-known/agent.json no corresponde a esta versión y devuelve 404.")
print("Un 404 en esa ruta alternativa no implica una falla del protocolo A2A.")
print("=" * 60)

**Antes de ejecutar 3.3 — Una variable cambia la topología**

**Objetivo.** Levantar el Planner y decidir, mediante configuración, si conoce al equipo completo.

En modo FULL TEAM, `SIMULATOR_AGENT_RESOURCE_NAME=local:8089` hace que el Planner agregue un `RemoteA2aAgent`. Sin esa variable funciona en modo SOLO con Planner + Evaluator. No cambiamos el binario: cambiamos la topología desplegada.

**Qué observar:** el log de arranque debe enumerar la tool del Evaluator y la conexión A2A al Simulator. El Runner no aparece como tool directa del Planner porque vive detrás del Simulator.

> 🧭 **Pausa de diseño:** ¿cuándo conviene que una capacidad sea configuración y no una bifurcación de código?

✅ **Señal de éxito:** `Planner escuchando en :8084` y `Las DOS tools cargaron`.

In [ ]:
# @title 3.3 — Levantar el Planner en modo FULL TEAM (:8084) { display-mode: "form" }
# La variable de entorno es TODA la diferencia entre Solo y Full Team.

MODO = "FULL TEAM (Planner + Evaluator + Simulator + Runner)"  # @param ["FULL TEAM (Planner + Evaluator + Simulator + Runner)", "SOLO (Planner + Evaluator)"]

planner_env = {**os.environ, "PYTHONUNBUFFERED": "1"}
if MODO.startswith("FULL"):
    planner_env["SIMULATOR_AGENT_RESOURCE_NAME"] = "local:8089"
else:
    planner_env.pop("SIMULATOR_AGENT_RESOURCE_NAME", None)

plan_log = f"{LOGS}/planner.log"

planner = subprocess.Popen(
    ["python", "-m", "src.planner_agent.runtime.local_server"],
    cwd=REPO_DIR, env=planner_env,
    stdout=open(plan_log, "w"), stderr=subprocess.STDOUT,
)
print(f"Planner arrancando (pid {planner.pid})...")
esperar_puerto(8084, "Planner", plan_log)

print("\n--- arranque del Planner ---")
log = open(plan_log).read()
for linea in log.splitlines():
    if any(k in linea for k in ("Mode:", "Executor:", "Agent:", "Tools:", "Added ", "  - ")):
        print(" ", linea.rstrip())

print("\n" + "=" * 60)
if "Added A2A Simulation Controller tool" in log:
    print("✓ Las DOS tools cargaron: Evaluator local + Simulator por A2A.")
else:
    print("· Solo el Evaluator. Sin SIMULATOR_AGENT_RESOURCE_NAME no hay tercer agente.")
print("=" * 60)

**Antes de ejecutar 3.4 — Una UI de desarrollo también es observabilidad**

**Objetivo.** Iniciar ADK Dev UI para conversar con el Planner e inspeccionar sus tool calls.

La UI no reemplaza a los agentes: es otro proceso que carga la aplicación y expone conversación, sesiones y trazas. En Colab, el servidor vive dentro de la VM y necesita una URL de acceso. El proxy firmado puede servir para una prueba rápida, pero en este workshop la vía confiable será el Quick Tunnel del paso siguiente.

**Qué observar:** primero validamos que el puerto 8000 responda desde la VM. Recién después intentamos abrir una interfaz. Esta secuencia separa “el servidor no levantó” de “el navegador no puede alcanzarlo”.

> 🧭 **Pausa de diseño:** si la pantalla queda negra pero `localhost:8000` responde, ¿falló el agente o falló la capa de acceso?

✅ **Señal de éxito:** la celda confirma el puerto. Si el iframe devuelve 403, continuá con 3.5: es el comportamiento esperado del proxy en algunas sesiones.

In [ ]:
# @title 3.4 — La UI: ADK Dev UI dentro de Colab { display-mode: "form" }
# `adk web` es la interfaz de desarrollo de ADK: un chat con el agente y, al
# lado, la traza de cada tool call, del sub-agente Evaluator y del salto A2A al
# Simulator. Corre en la VM y Colab intenta publicarla mediante su proxy.
#
# La URL NO es fija: Colab firma un subdominio por sesion y por usuario
# (https://8000-m-s-<hash>.<region>.prod.colab.dev). Por eso hay que pedirla en
# runtime con google.colab.kernel.proxyPort(8000). Copiarla de un tutorial no
# funciona nunca.

import os, shutil, subprocess, sys, time
import httpx
from google.colab import output
from IPython.display import HTML, display

ADK_PORT = 8000
UI_PATH = "/dev-ui/?app=planner_agent"

adk_env = {**os.environ, "SIMULATOR_AGENT_RESOURCE_NAME": "local:8089", "PYTHONUNBUFFERED": "1"}
# --host 0.0.0.0: escucha en todas las interfaces de la VM, no solo en loopback.
adk_bin = shutil.which("adk")
adk_cmd = ([adk_bin] if adk_bin else [sys.executable, "-m", "google.adk.cli"]) + \
          ["web", "--host", "0.0.0.0", "--port", str(ADK_PORT), "src"]

adkweb = subprocess.Popen(
    adk_cmd, cwd=REPO_DIR, env=adk_env,
    stdout=open(f"{LOGS}/adkweb.log", "w"), stderr=subprocess.STDOUT,
)
print(f"ADK Dev UI arrancando (pid {adkweb.pid}):\n  {' '.join(adk_cmd)}\n")

if not esperar_puerto(ADK_PORT, "ADK Dev UI", f"{LOGS}/adkweb.log", segundos=120):
    raise SystemExit("adk web no levantó. Revisá el log mostrado arriba.")

# Health check DESDE la VM antes de dar la URL: si esto responde, el servidor
# está bien y cualquier problema posterior es del proxy o de la cuenta.
apps = httpx.get(f"http://127.0.0.1:{ADK_PORT}/list-apps", timeout=10).json()
print(f"✓ el servidor responde en :{ADK_PORT} y ve los agentes: {apps}")

# IMPORTANTE: no imprimimos la URL firmada de proxyPort porque abrirla en una
# pestaña nueva carga el HTML pero devuelve 403 para los chunks JavaScript. La
# URL solo es válida dentro del contexto autenticado del notebook. Colab genera
# esa URL internamente para el iframe. Ver docs/GOTCHAS.md #21.
print("\nLa UI, acá abajo (agente `planner_agent` → escribí el pedido → panel Events):")
output.serve_kernel_port_as_iframe(ADK_PORT, path=UI_PATH, height="700")

print("\n" + "=" * 70)
print("NO ABRAS EL PROXY EN OTRA PESTAÑA: los chunks JavaScript reciben 403.")
print("SI ESTE IFRAME QUEDA EN BLANCO: pasá a la celda 4.3 o probá la 3.5.")
print("El proxy de Colab devuelve 403 en los chunks de JavaScript de la UI")
print("(el HTML carga, el JS no). No es tu servidor: el health check de arriba")
print("ya probó que responde. La 4.3 muestra la misma coreografía sin proxy.")
print("=" * 70)

**Antes de ejecutar 3.5 — Publicar la ADK Dev UI mediante un túnel**

En algunas sesiones de Colab, el proxy integrado entrega el HTML de la ADK Dev UI pero responde `403` al solicitar sus archivos JavaScript. En ese caso el servidor está sano, aunque la interfaz aparezca en blanco.

Para evitar esa capa, esta celda crea un Cloudflare Quick Tunnel hacia el puerto 8000 y devuelve una URL efímera. El mismo mecanismo se reutiliza en 4.4 para publicar la Race View en una segunda URL.

> ⚠️ **Seguridad:** la URL es pública y no tiene autenticación. Cualquier persona que la conozca puede enviar solicitudes al agente y consumir cuota de Vertex AI. No la compartas fuera del workshop y ejecutá la celda de limpieza al terminar.

Si preferís no publicar el agente, podés omitir el túnel y utilizar la traza de 4.3, que se comunica con la API desde el kernel de Colab.


In [ ]:
# @title 3.5 — Túnel público a la ADK Dev UI { display-mode: "form" }

ENTIENDO_QUE_ES_PUBLICO = False  # @param {type:"boolean"}

import httpx, os, re, socket, subprocess, time
from urllib.parse import urlparse
from IPython.display import HTML, display

if not ENTIENDO_QUE_ES_PUBLICO:
    raise SystemExit(
        "Marcá la casilla de arriba para confirmar que entendés que la URL es "
        "pública y sin autenticación. O usá la celda 4.3, que no expone nada."
    )

# Binario de cloudflared para la VM de Colab (linux x86_64).
if not os.path.exists("/content/cloudflared"):
    !wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /content/cloudflared
print(subprocess.run(["/content/cloudflared", "--version"], capture_output=True, text=True).stdout.strip())

def abrir_tunel_cloudflare(puerto, nombre, ruta_salud, archivo_log):
    # No retorna hasta comprobar tanto la resolución DNS como la respuesta HTTP.
    proceso = subprocess.Popen(
        ["/content/cloudflared", "tunnel", "--url", f"http://localhost:{puerto}", "--no-autoupdate"],
        stdout=open(archivo_log, "w"), stderr=subprocess.STDOUT,
    )
    url = None
    for _ in range(40):
        time.sleep(2)
        match = re.search(
            r"https://[a-z0-9-]+\.trycloudflare\.com",
            open(archivo_log).read(),
        )
        if match:
            url = match.group(0)
            break
    if not url:
        proceso.terminate()
        raise RuntimeError(
            f"No levantó el túnel de {nombre}. Log:\n{open(archivo_log).read()[-2000:]}"
        )

    host = urlparse(url).hostname
    ultimo_error = None
    for intento in range(30):
        if proceso.poll() is not None:
            raise RuntimeError(
                f"cloudflared para {nombre} terminó con código {proceso.returncode}.\n"
                f"{open(archivo_log).read()[-2000:]}"
            )
        try:
            socket.getaddrinfo(host, 443)
            respuesta = httpx.get(
                f"{url}{ruta_salud}", timeout=15, follow_redirects=True
            )
            respuesta.raise_for_status()
            print(f"✓ túnel de {nombre}: HTTP {respuesta.status_code}")
            return proceso, url, respuesta.text
        except (OSError, httpx.HTTPError) as error:
            ultimo_error = error
            print(f"Esperando {nombre}/DNS ({intento + 1}/30): {type(error).__name__}")
            time.sleep(3)
    proceso.terminate()
    raise RuntimeError(f"El túnel de {nombre} no respondió: {ultimo_error!r}")

tunel_log = f"{LOGS}/cloudflared-ui.log"
tunel_anterior = globals().get("tunel")
if tunel_anterior and tunel_anterior.poll() is None:
    tunel_anterior.terminate()
    tunel_anterior.wait(timeout=5)
tunel, url_publica, estado = abrir_tunel_cloudflare(
    ADK_PORT, "ADK Dev UI", "/list-apps", tunel_log
)
ui = f"{url_publica}/dev-ui/?app=planner_agent"
print(f"\n  {ui}\n")
display(HTML(
    f'<p style="font-size:1.2em"><a href="{ui}" target="_blank">🔗 Abrir la ADK Dev UI '
    f'(túnel público)</a></p><p style="color:#b00">No compartas esta URL. La celda '
    f'de Limpieza apaga este túnel y el de la carrera.</p>'))

**Antes de ejecutar 3.6 — Diseñar una solicitud que recorra toda la arquitectura**

Una prueba end-to-end debe obligar al sistema a utilizar cada capacidad relevante. Por eso el pedido incluye distancia certificada, punto de largada, logística, evaluación, simulación y experiencia de corredores.

Con esta solicitud verificamos que:

- el Planner utilice herramientas para calcular la ruta;
- el Evaluator puntúe el plan con su rúbrica;
- el Simulator sea alcanzado mediante A2A;
- el Simulator delegue el análisis de cohortes al Runner;
- la respuesta final identifique puntuaciones, veredictos y hallazgos.

Abrí una sesión nueva en la ADK Dev UI, seleccioná `planner_agent` y pegá el pedido completo de la próxima celda.


In [ ]:
# @title 3.6 — Copiar este pedido en la ADK Dev UI { display-mode: "form" }

PEDIDO_BUENOS_AIRES = """Plan a scenic marathon in Buenos Aires for 30,000 runners.

Create a certified-distance route of 26.2 miles (42.195 km), starting and finishing at the Obelisco. Use the route-planning tool and include the calculated waypoints, hydration stations, medical tents, traffic closures, community impact, logistics, finances, timeline, and risks.

Then send the complete plan to evaluator_agent for scoring and afterward to simulator_agent for the final readiness verdict. Ask simulator_agent to delegate runner experience to runner_agent for elite, competitive, main-pack, and back-of-pack cohorts. Include the evaluation scores, overall score, simulation verdict, runner readiness, and runner findings in the final response."""

print(PEDIDO_BUENOS_AIRES)
print("\n→ Copiá este texto, volvé a la ADK Dev UI y envialo a planner_agent.")

---
## Paso 4 — Ejecutar el sistema end-to-end

Comprobar que un servidor responde no demuestra colaboración entre agentes. En este paso enviamos una solicitud real y seguimos su recorrido completo.

La ejecución esperada es:

1. `list_skills` y `load_skill`: el Planner descubre conocimiento procedural.
2. `plan_marathon_route`: calcula waypoints y GeoJSON.
3. `add_water_stations` y `add_medical_tents`: calcula infraestructura.
4. El Planner redacta el plan operativo.
5. `evaluator_agent`: puntúa siete criterios mediante `AgentTool`.
6. `simulator_agent`: recibe el plan mediante A2A y aplica el gate de readiness.
7. `runner_agent`: analiza cuatro cohortes dentro del proceso del Simulator.
8. El Planner compone la respuesta final con plan, puntuaciones y veredictos.

La duración depende del modelo y de la disponibilidad de Vertex AI. El stream de eventos permite distinguir una espera legítima de una ejecución detenida.

> **Resultado del paso:** evidencia observable de la colaboración, no sólo una respuesta final.


**Antes de ejecutar 4.1 — Enviar una solicitud end-to-end**

**Objetivo.** Enviar un pedido real y comprobar que la colaboración completa produce un resultado útil.

Los tres parámetros se convierten en la intención del usuario. A partir de ahí el Planner debe descubrir Skills, llamar herramientas, redactar, pedir evaluación, cruzar A2A hacia el Simulator y recuperar la perspectiva del Runner. El script imprime eventos mientras ocurren para que la espera sea observable.

**Antes de ejecutar:** confirmá que 8084 y 8089 siguen activos. No interrumpas la celda si pasa varios segundos sin texto: los jueces LLM pueden estar trabajando.

> 🧭 **Pausa de diseño:** ¿qué evidencias necesitaríamos para afirmar que colaboraron cuatro agentes y no que un único modelo escribió todo?

✅ **Señal de éxito:** termina con `exit=0` y la respuesta incluye scores, overall, veredicto de simulación y hallazgos del Runner.

In [ ]:
# @title 4.1 — Mandar el pedido end-to-end (~1 min) { display-mode: "form" }

CIUDAD_PEDIDO = "Buenos Aires"  # @param {type:"string"}
PARTICIPANTES = 30000  # @param {type:"integer"}
TEMA = "scenic"  # @param ["scenic", "fast", "charity"]

import subprocess, sys, time

t0 = time.time()
proc = subprocess.Popen(
    [sys.executable, "scripts/send_request.py",
     "--city", CIUDAD_PEDIDO,
     "--participants", str(PARTICIPANTES),
     "--theme", TEMA,
     "--watch", "--timeout", "900"],
    cwd=REPO_DIR, env=planner_env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
for linea in proc.stdout:
    print(linea, end="")
proc.wait()
print(f"\n⏱  {time.time() - t0:.0f}s   exit={proc.returncode}")

**Antes de ejecutar 4.2 — Los logs convierten una respuesta en evidencia**

**Objetivo.** Reconstruir la coreografía desde los dos procesos sin depender de la interfaz visual.

Filtramos el log del Planner para encontrar tools, evaluación y A2A; después miramos el log del Simulator para encontrar la delegación al Runner. Esta segunda lectura es necesaria porque una frontera de observabilidad también es una frontera de proceso.

**Qué observar:** las tools determinísticas aparecen en el Planner, el salto A2A conecta con el Simulator y `simulate_runner_cohorts` aparece del otro lado. Un trace parcial no implica que el agente ausente no haya trabajado.

> 🧭 **Pausa de diseño:** ¿quién debería correlacionar trazas cuando los agentes se desplieguen en máquinas distintas?

✅ **Señal de éxito:** aparecen eventos bajo `TOOL`, `EVAL` y `A2A`, más actividad del Runner en `simulator.log`.

In [ ]:
# @title 4.2 — Qué pasó por dentro: la traza de los cuatro agentes { display-mode: "form" }
# Ejecutá esta celda después de 4.1 para reconstruir la coreografía.

import re

log = open(f"{LOGS}/planner.log").read()

interesantes = [
    ("TOOL",   r"plan_marathon_route|add_water_stations|add_medical_tents|load_skill|list_skills"),
    ("EVAL",   r"evaluate_plan|vertex_ai_eval|_evals_common|heuristic|evaluator_agent"),
    ("A2A",    r"agent-card|simulator_agent|Overriding agent card|a2a"),
    ("ERROR",  r"ERROR|Traceback"),
]

for linea in log.splitlines():
    for etiqueta, patron in interesantes:
        if re.search(patron, linea, re.I):
            print(f"[{etiqueta:5s}] {linea.strip()[:150]}")
            break

print("\n--- log del Simulator (acá también vive runner_agent) ---")
sim = open(f"{LOGS}/simulator.log").read()
for linea in sim.splitlines()[-80:]:
    if linea.strip() and re.search(r"runner_agent|simulate_runner_cohorts|function|tool|error", linea, re.I):
        print(" ", linea.strip()[:150])

**Antes de ejecutar 4.3 — La coreografía completa, evento por evento**

Esta vista se comunica con la API de `adk web` desde el kernel de Colab. Por eso puede mostrar la secuencia aunque el navegador no logre cargar la interfaz mediante el proxy.

La celda crea una sesión, envía el pedido, consume el stream de eventos y construye una tabla con tiempo, autor, acción, mecanismo y ubicación.

El stream del Planner muestra su primera frontera de delegación:

- el Evaluator se ejecuta en el mismo proceso mediante `AgentTool`;
- el Simulator responde por HTTP mediante A2A;
- el Runner se ejecuta dentro del proceso del Simulator.

Por ese motivo el Runner no aparece como una fila independiente en el stream del Planner. Su participación se comprueba en `simulator.log` y en `runner_findings`. Una traza sólo observa lo que ocurre dentro de su alcance.

> **Requisito:** la celda 3.4 debe haber iniciado `adk web` en el puerto 8000.


In [ ]:
# @title 4.3 — La coreografía completa, evento por evento { display-mode: "form" }

import json, time
import httpx
from IPython.display import HTML, display

PEDIDO = globals().get("PEDIDO_BUENOS_AIRES", '''Plan a scenic marathon in Buenos Aires for 30,000 runners.

Create a certified-distance route of 26.2 miles (42.195 km), starting and finishing at the Obelisco. Use the route-planning tool and include the calculated waypoints, hydration stations, medical tents, traffic closures, community impact, logistics, finances, timeline, and risks.

Then send the complete plan to evaluator_agent for scoring and afterward to simulator_agent for the final readiness verdict. Ask simulator_agent to delegate runner experience to runner_agent for elite, competitive, main-pack, and back-of-pack cohorts. Include the evaluation scores, overall score, simulation verdict, runner readiness, and runner findings in the final response.''')  # @param {type:"string"}

BASE = f"http://127.0.0.1:{ADK_PORT}"
sid = httpx.post(f"{BASE}/apps/planner_agent/users/user/sessions", json={}, timeout=30).json()["id"]
print(f"session: {sid}\nmandando el pedido... (~1 min)\n")

COLOR = {"planner_agent": "#FF6B35", "evaluator_agent": "#0077B6", "simulator_agent": "#2A9D8F"}
DONDE = {
    "list_skills": ("Skill", "en proceso"), "load_skill": ("Skill", "en proceso"),
    "plan_marathon_route": ("Tool", "en proceso · Dijkstra, sin LLM"),
    "add_water_stations": ("Tool", "en proceso · calculado"),
    "add_medical_tents": ("Tool", "en proceso · calculado"),
    "evaluator_agent": ("AgentTool", "MISMO proceso · 6 jueces LLM + 1 regex"),
    "simulator_agent": ("RemoteA2aAgent", "OTRO proceso · HTTP al :8089"),
}

filas, t0, final = [], time.time(), ""
with httpx.stream("POST", f"{BASE}/run_sse", timeout=900, json={
    "app_name": "planner_agent", "user_id": "user", "session_id": sid,
    "new_message": {"role": "user", "parts": [{"text": PEDIDO}]}, "streaming": False,
}) as resp:
    for linea in resp.iter_lines():
        if not linea.startswith("data:"):
            continue
        ev = json.loads(linea[5:])
        t = time.time() - t0
        for parte in (ev.get("content") or {}).get("parts", []):
            if "functionCall" in parte:
                n = parte["functionCall"]["name"]
                tipo, donde = DONDE.get(n, ("Tool", ""))
                filas.append((t, ev.get("author", "?"), "llama", n, tipo, donde))
                print(f"  [{t:5.1f}s] → {n}")
            elif "functionResponse" in parte:
                n = parte["functionResponse"]["name"]
                filas.append((t, ev.get("author", "?"), "responde", n, "", ""))
            elif parte.get("text"):
                final = parte["text"]
                filas.append((t, ev.get("author", "?"), "responde al usuario", "", "", f"{len(final)} caracteres"))

total = time.time() - t0
print(f"\n{len(filas)} eventos en {total:.0f}s\n")

html = ['<div style="font-family:system-ui;max-width:900px">',
        f'<h3 style="margin:0 0 4px">Una corrida completa · {len(filas)} eventos · {total:.0f}s</h3>',
        '<table style="border-collapse:collapse;width:100%;font-size:13px">']
for t, autor, verbo, nombre, tipo, donde in filas:
    c = COLOR.get(autor, "#888")
    destacar = "font-weight:700" if tipo in ("AgentTool", "RemoteA2aAgent") else ""
    html.append(
        f'<tr style="border-bottom:1px solid #eee">'
        f'<td style="padding:6px 10px;color:#999;white-space:nowrap">{t:5.1f}s</td>'
        f'<td style="padding:6px 10px"><span style="background:{c};color:#fff;padding:2px 8px;'
        f'border-radius:10px;white-space:nowrap">{autor}</span></td>'
        f'<td style="padding:6px 10px;color:#666">{verbo}</td>'
        f'<td style="padding:6px 10px;{destacar}">{nombre}</td>'
        f'<td style="padding:6px 10px;color:#666">{tipo}</td>'
        f'<td style="padding:6px 10px;color:#888">{donde}</td></tr>')
html.append('</table><p style="color:#666;font-size:13px;margin-top:10px">'
            'Las dos filas en negrita muestran la primera delegación: <b>evaluator_agent</b> corre dentro de este '
            'mismo proceso (AgentTool) y <b>simulator_agent</b> vive en otro, alcanzado por A2A sobre '
            'HTTP. Dentro del Simulator, <b>runner_agent</b> vuelve a usar AgentTool; su detalle queda del otro lado de la frontera A2A.</p></div>')
display(HTML("".join(html)))

print("=" * 70)
print("PLAN FINAL")
print("=" * 70)
print(final)

**Antes de ejecutar 4.4 — Buenos Aires Race View**

La ADK Dev UI permite observar decisiones y delegaciones. La Race View muestra las consecuencias operativas sobre la ciudad y los corredores.

La visualización combina dos resultados:

1. El **Planner** aporta el GeoJSON, los puestos de hidratación y la cobertura médica.
2. El **Runner** aporta cuatro cohortes con ritmo, llegada estimada, abandono proyectado y riesgo.

Los puntos animados son una muestra visual de 30.000 participantes; no representan llamadas individuales a un LLM. Color y velocidad codifican la cohorte. La tabla, el reloj y la animación consumen los mismos datos determinísticos, evitando resultados contradictorios.

La celda genera una página estática, inicia un servidor en el puerto 8010 y publica una segunda URL mediante Quick Tunnel:

- **ADK Dev UI:** conversación, herramientas y delegaciones;
- **Race View:** circuito, servicios y cohortes en movimiento.

La Race View es pública mientras el túnel esté activo, pero es de sólo lectura: no envía prompts ni consume Vertex AI.

> **Idea clave:** primero observamos las decisiones del sistema; después observamos su impacto sobre las personas.


In [ ]:
# @title 4.4 — Abrir la carrera animada de Buenos Aires { display-mode: "form" }

PARTICIPANTES_VISUAL = 30000  # @param {type:"integer"}
CORREDORES_VISUALES = 120  # @param {type:"slider", min:20, max:180, step:10}

import importlib.util, os, subprocess, sys

# Es autocontenida: funciona aunque hayas saltado el Paso 2.
if "route_tools" not in globals():
    spec = importlib.util.spec_from_file_location(
        "route_tools", f"{REPO_DIR}/src/planner_agent/skills/route-planning/tools.py"
    )
    route_tools = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(route_tools)

from src.planner_agent.visualization import build_animated_race_map
from src.runner_agent.agent.tools import simulate_runner_cohorts

ruta_visual = route_tools.plan_marathon_route(
    "Buenos Aires", start_landmark="Obelisco", target_distance_km=42.195
)
agua_visual = route_tools.add_water_stations(
    total_distance_km=42.195, participants=PARTICIPANTES_VISUAL
)
med_visual = route_tools.add_medical_tents(
    total_distance_km=42.195, participants=PARTICIPANTES_VISUAL
)

plan_para_corredores = globals().get("final") or globals().get("PLAN_COMPLETO") or (
    f"Buenos Aires marathon for {PARTICIPANTES_VISUAL:,} runners, 26.2 miles "
    "(42.195 km), wave starts, chip timing, hydration every 2.5 km, medical "
    "tents, ambulances and emergency access."
)
reporte_corredores = simulate_runner_cohorts(
    plan_para_corredores, participants=PARTICIPANTES_VISUAL
)

print("✓ GeoJSON del Planner cargado")
print(f"✓ {agua_visual['water_station_count']} puestos de hidratación")
print(f"✓ {med_visual['medical_tent_count']} puestos médicos")
print(f"✓ {CORREDORES_VISUALES} corredores animados representan {PARTICIPANTES_VISUAL:,} participantes")
print(f"✓ Runner Agent: {reporte_corredores['runner_readiness']} · "
      f"{reporte_corredores['projected_finishers']:,} llegadas proyectadas\n")

print(f"{'cohorte':18s} {'corredores':>10s} {'ritmo':>9s} {'llegada':>9s} {'riesgo':>9s}")
print("-" * 62)
for cohorte in reporte_corredores["cohorts"]:
    minutos = cohorte["projected_finish_minutes"]
    llegada = f"{minutos // 60:d}h {minutos % 60:02d}m"
    print(f"{cohorte['cohort']:18s} {cohorte['runners']:10,d} "
          f"{cohorte['pace_min_per_km']:7.2f}/km {llegada:>9s} {cohorte['risk']:>9s}")

print("\nHallazgos del Runner Agent:")
for hallazgo in reporte_corredores["findings"]:
    print(f"  · {hallazgo}")
print("\nUsá Pausar, Reiniciar y 1×/2×/4× en el panel del mapa.\n")

mapa_carrera = build_animated_race_map(
    ruta_visual,
    agua_visual,
    med_visual,
    participants=PARTICIPANTES_VISUAL,
    runner_count=CORREDORES_VISUALES,
    cohort_results=reporte_corredores["cohorts"],
)

# La vista final usa la misma vía que sí funcionó para la UI: servidor HTTP +
# Cloudflare Quick Tunnel. Exigimos haber aceptado el aviso de la celda 3.5.
if not globals().get("ENTIENDO_QUE_ES_PUBLICO") or "abrir_tunel_cloudflare" not in globals():
    raise SystemExit("Primero ejecutá la celda 3.5 y aceptá el aviso del túnel público.")

RACE_VIEW_PORT = 8010
RACE_VIEW_DIR = "/content/marathon-race-view"
os.makedirs(RACE_VIEW_DIR, exist_ok=True)
mapa_carrera.save(f"{RACE_VIEW_DIR}/index.html")

for proceso_anterior in (globals().get("race_server"), globals().get("race_tunnel")):
    if proceso_anterior and proceso_anterior.poll() is None:
        proceso_anterior.terminate()
        proceso_anterior.wait(timeout=5)

race_server_log = f"{LOGS}/race-view.log"
race_server = subprocess.Popen(
    [sys.executable, "-m", "http.server", str(RACE_VIEW_PORT),
     "--bind", "0.0.0.0", "--directory", RACE_VIEW_DIR],
    stdout=open(race_server_log, "w"), stderr=subprocess.STDOUT,
)
if not esperar_puerto(RACE_VIEW_PORT, "Race View", race_server_log, segundos=30):
    raise SystemExit("La vista de carrera no pudo levantar su servidor HTTP.")

race_tunnel, race_url, _ = abrir_tunel_cloudflare(
    RACE_VIEW_PORT, "Race View", "/", f"{LOGS}/cloudflared-race.log"
)
print(f"\n✓ Race View publicada:\n  {race_url}\n")
display(HTML(
    f'<div style="padding:14px;border:1px solid #ddd;border-radius:12px;margin:8px 0">'
    f'<a href="{race_url}" target="_blank" style="font-size:1.25em;font-weight:700">'
    f'🏃 Abrir Buenos Aires Race View</a><br>'
    f'<span style="color:#666">Abrila junto a la ADK Dev UI para mostrar decisiones y carrera.</span>'
    f'</div><iframe src="{race_url}" width="100%" height="720" '
    f'style="border:0;border-radius:12px" allowfullscreen></iframe>'))

---
## Paso 5 — Laboratorio de resiliencia

Un sistema confiable también debe explicar qué ocurre cuando una dependencia deja de estar disponible. Vamos a ejecutar dos experimentos controlados.

| Experimento | Cambio controlado | Capacidad que observamos |
|---|---|---|
| A — Evaluación degradada | `EVAL_MODE=heuristic` | Continuidad con una estrategia de menor calidad |
| B — Dependencia remota no disponible | Detener el Simulator | Comportamiento del Planner ante una falla A2A |

En ambos casos, formulá una hipótesis antes de ejecutar y comparala con el comportamiento observado. El objetivo no es provocar un error: es descubrir si la política de degradación está expresada con claridad.

> **Resultado del paso:** distinguir disponibilidad técnica, calidad del resultado y transparencia frente al usuario.


**Antes de ejecutar 5.A — Conservar disponibilidad sin el juez LLM**

**Hipótesis.** Si quitamos el juez LLM, el Planner debería seguir entregando el mismo tipo de respuesta mediante evaluación heurística.

La celda detiene solamente el Planner, conserva al Simulator, cambia `EVAL_MODE` y vuelve a ejecutar el mismo pedido. Es un experimento controlado: una variable cambia y el contrato externo permanece estable.

**Antes de mirar el resultado, predecí:** ¿será más rápido?, ¿los scores serán más extremos?, ¿seguirá apareciendo el Runner?

> 🧭 **Pausa de diseño:** ¿un fallback debe intentar ser igual de inteligente o priorizar disponibilidad y transparencia?

✅ **Señal de éxito:** el pedido termina, informa método heurístico y conserva el resto de la colaboración. Compará tiempo y calidad con 4.1.

In [ ]:
# @title 5.A — Forzar evaluación heurística { display-mode: "form" }

import signal, time

def frenar(proc, nombre):
    if proc and proc.poll() is None:
        proc.send_signal(signal.SIGINT)
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
        print(f"· {nombre} detenido")

frenar(planner, "Planner")
time.sleep(2)

planner_env["EVAL_MODE"] = "heuristic"
plan_log = f"{LOGS}/planner_heuristic.log"
planner = subprocess.Popen(
    ["python", "-m", "src.planner_agent.runtime.local_server"],
    cwd=REPO_DIR, env=planner_env,
    stdout=open(plan_log, "w"), stderr=subprocess.STDOUT,
)
esperar_puerto(8084, "Planner (heurístico)", plan_log)

t0 = time.time()
proc = subprocess.Popen(
    [sys.executable, "scripts/send_request.py", "--city", CIUDAD_PEDIDO,
     "--participants", str(PARTICIPANTES), "--theme", TEMA, "--watch", "--timeout", "600"],
    cwd=REPO_DIR, env=planner_env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
for linea in proc.stdout:
    print(linea, end="")
proc.wait()

print(f"\n⏱  {time.time() - t0:.0f}s  —  compará con la corrida con juez LLM.")
print("Mismo sistema, misma interfaz, sin una sola llamada al evaluador LLM.")

**Antes de ejecutar 5.B — Experimento: perder una dependencia remota**

**Hipótesis.** Si apagamos el Simulator, el Evaluator local todavía existe, pero desaparecen el gate remoto y el Runner que vive detrás de él.

Este experimento hace visible el costo de una frontera de red. `AgentTool` comparte destino de falla con el Planner; `RemoteA2aAgent` no. La respuesta final dependerá de cómo las instrucciones del orquestador definen una degradación aceptable.

**Antes de ejecutar:** elegí una hipótesis: falla total, plan parcial con advertencia o plan aparentemente completo. Después comparen la predicción con el comportamiento real.

> 🧭 **Pausa de diseño:** ¿qué debería prometer el Planner cuando no puede obtener un veredicto de readiness?

✅ **Aprendizaje esperado:** el puerto 8089 queda apagado y la salida revela si la política de error está realmente codificada en el orquestador.

In [ ]:
# @title 5.B — Apagar el Simulator y ver qué hace el Planner { display-mode: "form" }

frenar(simulador, "Simulator")
time.sleep(1)
print(f"Puerto 8089 vivo: {puerto_vivo(8089)}  (debería ser False)\n")

t0 = time.time()
proc = subprocess.Popen(
    [sys.executable, "scripts/send_request.py", "--city", CIUDAD_PEDIDO,
     "--participants", str(PARTICIPANTES), "--theme", TEMA, "--watch", "--timeout", "600"],
    cwd=REPO_DIR, env=planner_env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
for linea in proc.stdout:
    print(linea, end="")
proc.wait()
print(f"\n⏱  {time.time() - t0:.0f}s")

print("\n" + "=" * 60)
print("Pregunta de análisis: ¿entregó un plan, avisó que faltó el")
print("veredicto de simulación o interrumpió la solicitud?")
print("La respuesta depende de las instrucciones del Planner. Esto muestra")
print("que el prompt de un orquestador define política y comportamiento")
print("del sistema; no es solamente una tarea de redacción.")
print("=" * 60)

---
## Cierre — De una demostración a un sistema explicable

Construimos un flujo multiagente con decisiones observables:

- **Planner:** diseña y coordina;
- **Evaluator:** juzga mediante siete criterios;
- **Simulator:** aplica un gate de readiness;
- **Runner:** representa la experiencia de cuatro cohortes;
- **ADK:** define agentes, herramientas y delegación local;
- **A2A:** conecta procesos independientes;
- **Vertex AI Evaluation:** aporta rúbricas y explicaciones;
- **herramientas determinísticas:** calculan geometría, infraestructura y cohortes.

### Alcance de este workshop

La honestidad sobre el alcance también es parte del diseño:

- **Memory Bank:** el código contempla `VertexAiMemoryBankService`, pero sin `AGENT_ENGINE_ID` el workshop utiliza `InMemorySessionService`. No se demuestra persistencia entre sesiones.
- **Simulación de participantes:** el Runner modela cuatro cohortes y la Race View dibuja una muestra. No se ejecutan 30.000 agentes LLM individuales.
- **Infraestructura:** los servicios y túneles viven en la VM temporal de Colab. No se realiza un despliegue persistente.

Estas decisiones mantienen el ejercicio reproducible y permiten concentrarse en arquitectura, evaluación y observabilidad.

### Próximos pasos

| Recurso | Uso recomendado |
|---|---|
| [Codelab de referencia](https://codelabs.developers.google.com/next26/dev-keynote/build-multi-agent-marathon-planner) | Comparar el recorrido original |
| [Repositorio del workshop](https://github.com/jorgeucano/Multi-Agent-Marathon-Planner-Workshop) | Código, tags y documentación |
| `docs/GOTCHAS.md` | Diagnóstico de problemas conocidos |
| `docs/WORKSHOP.md` | Guía cronometrada de 60 minutos |
| [Implementación completa de Race Condition](https://github.com/GoogleCloudPlatform/race-condition) | Explorar gateway, Redis, WebSockets y frontend 3D |

Ejecutá la celda de limpieza antes de cerrar el notebook para detener servidores y túneles.


**Antes de ejecutar Limpieza — Cerrar también es parte del ciclo de vida**

**Objetivo.** Detener servidores locales y revocar los accesos temporales publicados por Cloudflare.

Durante el workshop levantamos cuatro procesos: Simulator, Planner, ADK Dev UI y Race View, además de dos túneles. Aunque la VM de Colab finalmente desaparece, cerrar explícitamente enseña responsabilidad operativa y evita confundir una segunda ejecución con procesos anteriores.

**Qué observar:** todos los puertos deben quedar en `False`. Esta celda no borra recursos de Google Cloud porque el workshop no desplegó servicios persistentes.

> 🧭 **Pausa de diseño:** ¿qué cambiaría en esta limpieza si hubiéramos desplegado Agent Engine, Cloud Run o una base de memoria persistente?

In [ ]:
# @title Limpieza — apagar servidores y túneles { display-mode: "form" }

import signal, subprocess

def detener_proceso(proc, nombre):
    if proc and proc.poll() is None:
        proc.send_signal(signal.SIGINT)
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
        print(f"· {nombre} detenido")

for proc, nombre in [(globals().get("planner"), "Planner"),
                    (globals().get("simulador"), "Simulator"),
                    (globals().get("adkweb"), "ADK Dev UI"),
                    (globals().get("tunel"), "túnel público de UI"),
                    (globals().get("race_server"), "servidor Race View"),
                    (globals().get("race_tunnel"), "túnel público de Race View")]:
    detener_proceso(proc, nombre)

!pkill -f "src.planner_agent.runtime.local_server" 2>/dev/null
!pkill -f "src.simulator_agent.runtime.local_server" 2>/dev/null
!pkill -f "adk.*web" 2>/dev/null
!pkill -f "http.server 8010" 2>/dev/null
!pkill -f cloudflared 2>/dev/null

import time
time.sleep(1)
print(f"\nPuerto 8084 vivo: {puerto_vivo(8084)}")
print(f"Puerto 8000 vivo: {puerto_vivo(8000)}")
print(f"Puerto 8089 vivo: {puerto_vivo(8089)}")
print(f"Puerto 8010 vivo: {puerto_vivo(8010)}")
print("\nNo hay nada más que borrar: este workshop no crea recursos persistentes")
print("en Google Cloud (ni Cloud Run, ni Agent Engine). Solo llamadas a la API.")
print("La VM de Colab se libera sola cuando cerrás la sesión.")